[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/01_data/01_proyecto_pipeline_datos.ipynb)

# 🛠️ Proyecto 01 · El pipeline de datos reproducible de CineMatch

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | 2,5 h |
| **GPU** | No necesaria. ≈ 0 unidades de Colab |
| **Prerrequisitos** | Lección 01 |

## 🎬 Contexto de negocio
La home de popularidad del proyecto 00 ya está en producción. Ahora el equipo va a empezar a entrenar modelos de verdad y la VP de Data te pide **una única fuente de verdad** para todos los experimentos: un pipeline que, dada una configuración, produzca **siempre los mismos** train/val/test, los guarde en parquet con sus metadatos y verifique que no hay leakage. Si dos científicos de datos comparan modelos, deben hacerlo sobre exactamente los mismos datos.

Este pipeline es el que usarán **todos los módulos siguientes del curso** (`cinematch_data.py`).

## 📦 Dataset
**MovieLens-100K** (`SCALE="small"`) o **MovieLens-1M** (`SCALE="full"`) de GroupLens: <https://files.grouplens.org/datasets/movielens/>. Positivo implícito = rating ≥ 4.

## ✅ Entregables y rúbrica
| # | Entregable | Criterio de aceptación |
|---|---|---|
| 1 | `PipelineConfig` | dataclass con todos los parámetros; el `hash` de la config identifica el dataset |
| 2 | `build_dataset(cfg)` | carga → dedup → implícito → split temporal global → k-core **solo en train** → val/test con usuarios e ítems conocidos |
| 3 | Validaciones | `validar(splits)` pasa: sin solapamiento temporal, sin duplicados, sin usuarios/ítems fríos en val/test, tipos correctos |
| 4 | Persistencia | parquet + `metadata.json` (config, tamaños, rangos temporales, sha256 de cada fichero); dos ejecuciones → mismos hashes |
| 5 | Baselines | aleatorio, popularidad, popularidad reciente (ventana ajustada **en val**), por segmento; Recall@10 y NDCG@10 en test. **Popularidad ≥ 5× aleatorio** |
| 6 | Informe de leakage | misma popularidad evaluada con split aleatorio y con popularidad "contaminada" por el test |
| 7 | Módulo | tu pipeline coincide con `cinematch_data.prepare_cinematch` (mismos tamaños) |

In [ ]:
!pip install -q pandas pyarrow scipy matplotlib

In [ ]:
import hashlib
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
SCALE = "small"     # "small" → 100k · "full" → 1m
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
#@title 🔧 Utilidades del curso: cinematch_data.py { display-mode: "form" }
# Escribe los módulos reutilizables del curso en el directorio actual.
# Última versión en GitHub (alternativa):
#   !wget -q https://raw.githubusercontent.com/padillacm/IA-practice/master/recsys-course/01_data/cinematch_data.py
_SOURCES = {}
_SOURCES['cinematch_data.py'] = '"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).\n\nInterfaz estable que reutilizan los módulos posteriores:\n\n    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"\n    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)\n    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)\n    inter = filter_k_core(inter, min_user=5, min_item=5)\n    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)\n    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test\n    train, test = random_split(inter, test_frac=0.2)\n    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse\n    recs = recommend_popular(train, users=test.user_id.unique(), k=10)\n    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado\n\nConvenciones de columnas: user_id (int), item_id (int), rating (float),\ntimestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").\n\nSi la descarga de GroupLens falla (sin red), `load_movielens` genera un dataset\nsintético con el mismo esquema (`make_synthetic_movielens`) y avisa.\n"""\nfrom __future__ import annotations\n\nimport io\nimport urllib.request\nimport warnings\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nBASE_URL = "https://files.grouplens.org/datasets/movielens"\nSIZES = {\n    "100k": "ml-100k",\n    "1m": "ml-1m",\n    "latest-small": "ml-latest-small",\n    "25m": "ml-25m",\n    "32m": "ml-32m",\n}\nGENRES_100K = [\n    "unknown", "Action", "Adventure", "Animation", "Children\'s", "Comedy", "Crime",\n    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",\n    "Romance", "Sci-Fi", "Thriller", "War", "Western",\n]\n\n\n# ----------------------------------------------------------------------------\n# Descarga y carga\n# ----------------------------------------------------------------------------\ndef download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:\n    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""\n    if size not in SIZES:\n        raise ValueError(f"size debe ser uno de {list(SIZES)}")\n    name = SIZES[size]\n    data_dir = Path(data_dir)\n    folder = data_dir / name\n    if folder.exists():\n        return folder\n    data_dir.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE_URL}/{name}.zip"\n    print(f"Descargando {url} ...")\n    with urllib.request.urlopen(url, timeout=60) as resp:\n        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)\n    return folder\n\n\ndef _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:\n    year = titles.str.extract(r"\\((\\d{4})\\)\\s*$")[0].astype("float")\n    clean = titles.str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True)\n    return clean, year\n\n\ndef _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if size == "100k":\n        ratings = pd.read_csv(folder / "u.data", sep="\\t",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K\n        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")\n        flags = raw[GENRES_100K].values.astype(bool)\n        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]\n        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),\n                              "genres": genres})\n    elif size == "1m":\n        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",\n                            names=["item_id", "title", "genres"], encoding="latin-1")\n    else:\n        ratings = pd.read_csv(folder / "ratings.csv").rename(\n            columns={"userId": "user_id", "movieId": "item_id"})\n        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})\n    items["title"], items["year"] = _split_title_year(items["title"].astype(str))\n    return ratings, items[["item_id", "title", "year", "genres"]]\n\n\ndef load_movielens(size: str = "100k", data_dir: str | Path = "data",\n                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Devuelve (ratings, items) con el esquema estándar de CineMatch.\n\n    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.\n    items:   item_id, title, year, genres ("Action|Comedy").\n    """\n    try:\n        folder = download_movielens(size, data_dir)\n        ratings, items = _read_folder(size, folder)\n    except Exception as e:  # sin red, URL caída, etc.\n        if not synthetic_fallback:\n            raise\n        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "\n                      "Usando datos SINTÉTICOS con el mismo esquema.")\n        ratings, items = make_synthetic_movielens()\n    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",\n                              "rating": "float32", "timestamp": "int64"})\n    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")\n    return ratings.reset_index(drop=True), items.reset_index(drop=True)\n\n\ndef load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:\n    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""\n    try:\n        folder = download_movielens(size, data_dir)\n    except Exception:\n        return None\n    if size == "100k":\n        return pd.read_csv(folder / "u.user", sep="|",\n                           names=["user_id", "age", "gender", "occupation", "zip"])\n    if size == "1m":\n        return pd.read_csv(folder / "users.dat", sep="::", engine="python",\n                           names=["user_id", "gender", "age", "occupation", "zip"])\n    return None\n\n\ndef make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,\n                             n_ratings: int = 100_000, seed: int = 42\n                             ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,\n    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""\n    rng = np.random.default_rng(seed)\n    genres = GENRES_100K[1:]\n    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K\n    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada\n    item_genre = rng.integers(0, len(genres), n_items)\n    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1\n    rng.shuffle(item_pop)\n    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,\n                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos\n    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta\n    activity = rng.lognormal(0, 1.0, n_users)\n    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)\n    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)\n    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)\n    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras\n    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes\n    rows = []\n    for u in range(n_users):\n        start = user_start[u]\n        end = min(t1, start + rng.uniform(1, 120) * 86_400)\n        avail = item_arrival <= end\n        n = int(min(activity[u], avail.sum() // 2))\n        ts = np.sort(rng.uniform(start, end, n))\n        age_days = np.maximum(0, end - item_arrival) / 86_400\n        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento\n        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail\n        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())\n        ts = np.maximum(ts, item_arrival[chosen] + 3600)\n        aff = taste[u][item_genre[chosen]] * len(genres)\n        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]\n                             + rng.normal(0, 0.7, n)), 1, 5)\n        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,\n                                  "rating": r, "timestamp": ts.astype(np.int64)}))\n    ratings = pd.concat(rows, ignore_index=True)\n    second = rng.integers(0, len(genres), n_items)\n    items = pd.DataFrame({\n        "item_id": np.arange(1, n_items + 1),\n        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],\n        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),\n        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"\n                   for a, b in zip(item_genre, second)],\n    })\n    return ratings, items\n\n\n# ----------------------------------------------------------------------------\n# Limpieza y señales\n# ----------------------------------------------------------------------------\ndef to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:\n    """Convierte ratings explícitos en interacciones implícitas positivas\n    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""\n    out = ratings[ratings["rating"] >= threshold].copy()\n    return out.reset_index(drop=True)\n\n\ndef dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:\n    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""\n    return (df.sort_values("timestamp", kind="stable")\n              .drop_duplicates(["user_id", "item_id"], keep=keep)\n              .reset_index(drop=True))\n\n\ndef filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,\n                  max_iter: int = 100) -> pd.DataFrame:\n    """k-core iterativo: repite hasta que todo usuario tenga >= min_user\n    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""\n    for _ in range(max_iter):\n        n0 = len(df)\n        ic = df["item_id"].map(df["item_id"].value_counts())\n        df = df[ic >= min_item]\n        uc = df["user_id"].map(df["user_id"].value_counts())\n        df = df[uc >= min_user]\n        if len(df) == n0:\n            break\n    return df.reset_index(drop=True)\n\n\n# ----------------------------------------------------------------------------\n# Splits\n# ----------------------------------------------------------------------------\ndef _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:\n    if users:\n        other = other[other["user_id"].isin(train["user_id"].unique())]\n    if items:\n        other = other[other["item_id"].isin(train["item_id"].unique())]\n    return other.reset_index(drop=True)\n\n\ndef random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,\n                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""\n    mask = np.random.default_rng(seed).random(len(df)) < test_frac\n    train, test = df[~mask].reset_index(drop=True), df[mask]\n    return train, _drop_cold(train, test, drop_cold, drop_cold)\n\n\ndef leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1\n                        ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.\n    Ojo: no hay un corte temporal global → el train de un usuario puede contener\n    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""\n    df = df.sort_values(["user_id", "timestamp"], kind="stable")\n    rank_from_end = df.groupby("user_id").cumcount(ascending=False)\n    n_user = df.groupby("user_id")["item_id"].transform("size")\n    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)\n    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)\n\n\ndef temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,\n                   drop_cold_users: bool = True, drop_cold_items: bool = True\n                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo\n    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a\n    producción: entrenas con todo lo anterior a T y sirves después de T.\n    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""\n    ts = df["timestamp"]\n    t_test = ts.quantile(1 - test_frac)\n    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test\n    train = df[ts < t_val].reset_index(drop=True)\n    val = df[(ts >= t_val) & (ts < t_test)]\n    test = df[ts >= t_test]\n    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)\n    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)\n    seen = pd.concat([train, val])\n    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)\n    return train, val, test\n\n\n# ----------------------------------------------------------------------------\n# Matrices dispersas\n# ----------------------------------------------------------------------------\ndef build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:\n    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""\n    users = np.sort(df["user_id"].unique())\n    items = np.sort(df["item_id"].unique())\n    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})\n\n\ndef to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,\n           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:\n    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).\n    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""\n    if user2idx is None or item2idx is None:\n        user2idx, item2idx = build_mappings(df)\n    r = df["user_id"].map(user2idx)\n    c = df["item_id"].map(item2idx)\n    ok = r.notna() & c.notna()\n    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \\\n        else df.loc[ok, value_col].to_numpy(np.float32)\n    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),\n                      shape=(len(user2idx), len(item2idx)))\n    X.sum_duplicates()\n    return X, user2idx, item2idx\n\n\n# ----------------------------------------------------------------------------\n# Baselines\n# ----------------------------------------------------------------------------\ndef _seen_by_user(train: pd.DataFrame) -> dict:\n    return train.groupby("user_id")["item_id"].agg(set).to_dict()\n\n\ndef _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:\n    out = []\n    for it in ranked:\n        if it not in seen:\n            out.append(int(it))\n            if len(out) == k:\n                break\n    return out\n\n\ndef popularity_ranking(train: pd.DataFrame, window_days: float | None = None,\n                       half_life_days: float | None = None) -> np.ndarray:\n    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.\n    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""\n    df = train\n    t_max = df["timestamp"].max()\n    if window_days is not None:\n        df = df[df["timestamp"] >= t_max - window_days * 86_400]\n    if half_life_days is not None:\n        age = (t_max - df["timestamp"]) / 86_400\n        w = 0.5 ** (age / half_life_days)\n        scores = w.groupby(df["item_id"]).sum()\n    else:\n        scores = df["item_id"].value_counts()\n    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()\n\n\ndef recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,\n                      window_days: float | None = None,\n                      half_life_days: float | None = None) -> dict:\n    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""\n    ranked = popularity_ranking(train, window_days, half_life_days)\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}\n\n\ndef recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,\n                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:\n    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).\n    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""\n    seg = train["user_id"].map(segment_of)\n    global_rank = popularity_ranking(train)\n    ranks = {}\n    for s, g in train.groupby(seg):\n        if len(g) >= min_segment_events:\n            r = g["item_id"].value_counts().index.to_numpy()\n            # completar con el ranking global para no quedarnos cortos\n            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)\n            for u in users}\n\n\ndef recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,\n                     exclude_seen: bool = True) -> dict:\n    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""\n    rng = np.random.default_rng(seed)\n    items = train["item_id"].unique()\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}\n\n\n# ----------------------------------------------------------------------------\n# Persistencia\n# ----------------------------------------------------------------------------\ndef save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:\n    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, f in frames.items():\n        f.to_parquet(out_dir / f"{name}.parquet", index=False)\n    return out_dir\n\n\ndef load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:\n    out_dir = Path(out_dir)\n    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names\n            if (out_dir / f"{n}.parquet").exists()}\n\n\n# ----------------------------------------------------------------------------\n# Pipeline completo de CineMatch (una llamada)\n# ----------------------------------------------------------------------------\ndef prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,\n                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,\n                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:\n    """Pipeline estándar del curso:\n    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal\n    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test\n    restringidos a usuarios e ítems conocidos en train.\n\n    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si\n    out_dir no es None, los guarda en parquet."""\n    ratings, items = load_movielens(size, data_dir)\n    inter = dedup_interactions(ratings)\n    if threshold is not None:\n        inter = to_implicit(inter, threshold)\n    train, val, test = temporal_split(inter, val_frac, test_frac,\n                                      drop_cold_users=False, drop_cold_items=False)\n    train = filter_k_core(train, min_user, min_item)\n    val = _drop_cold(train, val, True, True)\n    test = _drop_cold(train, test, True, True)\n    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}\n    if out_dir is not None:\n        save_splits(out_dir, train=train, val=val, test=test, items=items)\n    return out\n'
for _name, _src in _SOURCES.items():
    with open(_name, 'w', encoding='utf-8') as _fh:
        _fh.write(_src)
    print('✔ escrito', _name)

Del módulo `cinematch_data` usamos solo el **cargador** (`load_movielens`, con fallback sintético si no hay red). El resto lo construyes tú.

In [ ]:
from cinematch_data import load_movielens

ratings, items = load_movielens("100k" if SCALE == "small" else "1m")
ratings.head()

---
### TODO 1 — Configuración
Crea una `dataclass` (congelada) con: `size`, `threshold` (4.0), `min_user` (5), `min_item` (5), `val_frac` (0.1), `test_frac` (0.1), `out_dir` (`"data/cinematch"`). Añade un método `hash()` que devuelva los 10 primeros caracteres del sha256 del JSON de la config (sin `out_dir`).

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    # TODO: campos con sus valores por defecto
    ...

    def hash(self) -> str:
        # TODO
        raise NotImplementedError

cfg = PipelineConfig(size="100k" if SCALE == "small" else "1m")
cfg

### TODO 2 — Las piezas
Implementa (sin mirar `cinematch_data.py`):
- `dedup(df)`: un único evento por (usuario, ítem), el más reciente.
- `k_core(df, min_user, min_item)`: iterativo hasta converger.
- `temporal_split(df, val_frac, test_frac)`: cortes globales por cuantiles de `timestamp`; devuelve `(train, val, test)` con `train < t_val <= val < t_test <= test`.
- `drop_cold(train, other)`: elimina de `other` usuarios o ítems que no estén en `train`.

In [ ]:
def dedup(df: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

def k_core(df: pd.DataFrame, min_user: int, min_item: int) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

def temporal_split(df: pd.DataFrame, val_frac: float, test_frac: float):
    # TODO
    raise NotImplementedError

def drop_cold(train: pd.DataFrame, other: pd.DataFrame) -> pd.DataFrame:
    # TODO
    raise NotImplementedError

### TODO 3 — `build_dataset` y `validar`
`build_dataset(cfg, ratings)` encadena: dedup → filtro implícito (rating ≥ threshold) → split temporal → k-core en train → `drop_cold` de val y test. Devuelve un dict `{"train", "val", "test"}`.

`validar(splits)` debe lanzar `AssertionError` con un mensaje claro si algo falla:
1. `train.timestamp.max() < val.timestamp.min()` y `val.timestamp.max() <= test.timestamp.min()`.
2. Ningún par (usuario, ítem) duplicado dentro de un split ni entre splits.
3. Todo usuario e ítem de val/test existe en train.
4. En train todo usuario tiene ≥ `min_user` e ítem ≥ `min_item` interacciones.
5. Columnas `user_id, item_id, rating, timestamp` con tipos enteros/float.

In [ ]:
def build_dataset(cfg: PipelineConfig, ratings: pd.DataFrame) -> dict:
    # TODO
    raise NotImplementedError

def validar(splits: dict, cfg: PipelineConfig) -> None:
    # TODO
    raise NotImplementedError

splits = build_dataset(cfg, ratings)
validar(splits, cfg)
{k: v.shape for k, v in splits.items()}

### TODO 4 — Persistencia reproducible
`guardar(splits, cfg)` escribe `train/val/test/items.parquet` en `cfg.out_dir / cfg.hash()` y un `metadata.json` con: la config, nº de filas/usuarios/ítems por split, rango temporal (ISO) por split y el sha256 de cada parquet. Comprueba que **dos ejecuciones** producen los mismos hashes.

In [ ]:
def sha256(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def guardar(splits: dict, items: pd.DataFrame, cfg: PipelineConfig) -> dict:
    # TODO: devuelve el dict de metadatos
    raise NotImplementedError

meta1 = guardar(splits, items, cfg)
meta2 = guardar(build_dataset(cfg, ratings), items, cfg)
assert meta1["sha256"] == meta2["sha256"], "¡El pipeline no es determinista!"
print(json.dumps(meta1, indent=2, ensure_ascii=False)[:800])

### TODO 5 — Baselines y evaluación
Implementa `recall_at_k` y `ndcg_at_k` (binario; $\text{DCG}=\sum_{p=1}^{K} \frac{\mathbb{1}[\text{rel}]}{\log_2(p+1)}$, normalizado por el ideal) promediados sobre los usuarios de test, y los recomendadores:
- `rec_aleatorio`, `rec_popular(train, users, k, window_days=None)`, `rec_segmento(train, users, segmento_de, k)`.

Protocolo: ajusta `window_days ∈ {7, 14, 30, 60, 90, None}` maximizando Recall@10 en **val** entrenando con train; después **reentrena con train+val** y evalúa todos en **test**. Excluye siempre lo ya visto.

In [ ]:
def recall_at_k(recs: dict, test: pd.DataFrame, k: int = 10) -> float:
    # TODO
    raise NotImplementedError

def ndcg_at_k(recs: dict, test: pd.DataFrame, k: int = 10) -> float:
    # TODO
    raise NotImplementedError

def rec_aleatorio(train, users, k=10, seed=SEED) -> dict:
    # TODO
    raise NotImplementedError

def rec_popular(train, users, k=10, window_days=None) -> dict:
    # TODO
    raise NotImplementedError

def rec_segmento(train, users, segmento_de: dict, k=10) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: ajuste de ventana en val, tabla final en test (DataFrame: filas = baseline, columnas = Recall@10, NDCG@10)
tabla = ...
tabla

### TODO 6 — Informe de leakage
Evalúa `rec_popular` (sin ventana) en tres escenarios con Recall@10: (a) tu split temporal, (b) un split **aleatorio** 80/20 sobre las mismas interacciones implícitas, (c) split temporal pero con la popularidad calculada sobre **train + test**. Dibuja un gráfico de barras y explica en 3 frases qué ves.

In [ ]:
# TODO

---
## ⛔ SPOILER — Solución de referencia

Intenta resolverlo primero.

In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    size: str = "100k"
    threshold: float = 4.0
    min_user: int = 5
    min_item: int = 5
    val_frac: float = 0.1
    test_frac: float = 0.1
    out_dir: str = "data/cinematch"

    def hash(self) -> str:
        d = {k: v for k, v in asdict(self).items() if k != "out_dir"}
        return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:10]

cfg = PipelineConfig(size="100k" if SCALE == "small" else "1m")
print(cfg, "→ hash", cfg.hash())

In [ ]:
def dedup(df: pd.DataFrame) -> pd.DataFrame:
    return (df.sort_values("timestamp", kind="stable")
              .drop_duplicates(["user_id", "item_id"], keep="last").reset_index(drop=True))

def k_core(df: pd.DataFrame, min_user: int, min_item: int) -> pd.DataFrame:
    while True:
        n0 = len(df)
        df = df[df.item_id.map(df.item_id.value_counts()) >= min_item]
        df = df[df.user_id.map(df.user_id.value_counts()) >= min_user]
        if len(df) == n0:
            return df.reset_index(drop=True)

def temporal_split(df: pd.DataFrame, val_frac: float, test_frac: float):
    t_test = df.timestamp.quantile(1 - test_frac)
    t_val = df.timestamp.quantile(1 - test_frac - val_frac)
    train = df[df.timestamp < t_val]
    val = df[(df.timestamp >= t_val) & (df.timestamp < t_test)]
    test = df[df.timestamp >= t_test]
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

def drop_cold(train: pd.DataFrame, other: pd.DataFrame) -> pd.DataFrame:
    ok = other.user_id.isin(train.user_id.unique()) & other.item_id.isin(train.item_id.unique())
    return other[ok].reset_index(drop=True)

In [ ]:
def build_dataset(cfg: PipelineConfig, ratings: pd.DataFrame) -> dict:
    inter = dedup(ratings)
    inter = inter[inter.rating >= cfg.threshold]
    train, val, test = temporal_split(inter, cfg.val_frac, cfg.test_frac)
    train = k_core(train, cfg.min_user, cfg.min_item)
    return {"train": train, "val": drop_cold(train, val), "test": drop_cold(train, test)}

def validar(splits: dict, cfg: PipelineConfig) -> None:
    tr, va, te = splits["train"], splits["val"], splits["test"]
    assert tr.timestamp.max() < va.timestamp.min(), "train y val se solapan en el tiempo"
    assert va.timestamp.max() <= te.timestamp.min(), "val y test se solapan en el tiempo"
    todos = pd.concat([tr, va, te])
    assert not todos.duplicated(["user_id", "item_id"]).any(), "pares (usuario, ítem) duplicados"
    for nombre, d in [("val", va), ("test", te)]:
        assert d.user_id.isin(tr.user_id).all(), f"usuarios fríos en {nombre}"
        assert d.item_id.isin(tr.item_id).all(), f"ítems fríos en {nombre}"
    assert tr.user_id.value_counts().min() >= cfg.min_user, "k-core de usuarios no cumplido"
    assert tr.item_id.value_counts().min() >= cfg.min_item, "k-core de ítems no cumplido"
    for d in (tr, va, te):
        assert list(d.columns[:4]) == ["user_id", "item_id", "rating", "timestamp"]
        assert pd.api.types.is_integer_dtype(d.user_id) and pd.api.types.is_integer_dtype(d.timestamp)
    print("✔ validación superada")

splits = build_dataset(cfg, ratings)
validar(splits, cfg)
{k: v.shape for k, v in splits.items()}

In [ ]:
def sha256(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def guardar(splits: dict, items: pd.DataFrame, cfg: PipelineConfig) -> dict:
    out = Path(cfg.out_dir) / cfg.hash()
    out.mkdir(parents=True, exist_ok=True)
    meta = {"config": asdict(cfg), "hash": cfg.hash(), "splits": {}, "sha256": {}}
    for nombre, d in {**splits, "items": items}.items():
        path = out / f"{nombre}.parquet"
        d.to_parquet(path, index=False)
        meta["sha256"][nombre] = sha256(path)
        if "timestamp" in d:
            meta["splits"][nombre] = {
                "filas": len(d), "usuarios": int(d.user_id.nunique()), "items": int(d.item_id.nunique()),
                "desde": pd.to_datetime(d.timestamp.min(), unit="s").isoformat(),
                "hasta": pd.to_datetime(d.timestamp.max(), unit="s").isoformat()}
    (out / "metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    return meta

meta1 = guardar(splits, items, cfg)
meta2 = guardar(build_dataset(cfg, ratings), items, cfg)
assert meta1["sha256"] == meta2["sha256"], "¡El pipeline no es determinista!"
print(json.dumps(meta1["splits"], indent=2, ensure_ascii=False))

In [ ]:
def _rel(test):
    return test.groupby("user_id").item_id.agg(set)

def recall_at_k(recs: dict, test: pd.DataFrame, k: int = 10) -> float:
    return float(np.mean([len(r & set(recs.get(u, [])[:k])) / len(r) for u, r in _rel(test).items()]))

def ndcg_at_k(recs: dict, test: pd.DataFrame, k: int = 10) -> float:
    vals = []
    for u, r in _rel(test).items():
        rec = recs.get(u, [])[:k]
        dcg = sum(1 / np.log2(p + 2) for p, i in enumerate(rec) if i in r)
        idcg = sum(1 / np.log2(p + 2) for p in range(min(k, len(r))))
        vals.append(dcg / idcg)
    return float(np.mean(vals))

def _topk(ranking, vistos, k):
    return [int(i) for i in ranking[: k + len(vistos)] if i not in vistos][:k]

def rec_aleatorio(train, users, k=10, seed=SEED) -> dict:
    r = np.random.default_rng(seed); cat = train.item_id.unique()
    vistos = train.groupby("user_id").item_id.agg(set)
    return {u: _topk(r.permutation(cat), vistos.get(u, set()), k) for u in users}

def rec_popular(train, users, k=10, window_days=None) -> dict:
    d = train if window_days is None else train[train.timestamp >= train.timestamp.max() - window_days * 86_400]
    ranking = np.concatenate([d.item_id.value_counts().index.to_numpy(),
                              train.item_id.value_counts().index.to_numpy()])   # relleno con la global
    ranking = pd.unique(ranking)
    vistos = train.groupby("user_id").item_id.agg(set)
    return {u: _topk(ranking, vistos.get(u, set()), k) for u in users}

def rec_segmento(train, users, segmento_de: dict, k=10) -> dict:
    glob = train.item_id.value_counts().index.to_numpy()
    seg = train.user_id.map(segmento_de)
    rank_seg = {s: pd.unique(np.concatenate([g.item_id.value_counts().index.to_numpy(), glob]))
                for s, g in train.groupby(seg)}
    vistos = train.groupby("user_id").item_id.agg(set)
    return {u: _topk(rank_seg.get(segmento_de.get(u), glob), vistos.get(u, set()), k) for u in users}

In [ ]:
from cinematch_data import load_users

tr, va, te = splits["train"], splits["val"], splits["test"]
ventanas = [7, 14, 30, 60, 90, None]
res_val = {w: recall_at_k(rec_popular(tr, va.user_id.unique(), 10, w), va) for w in ventanas}
w_best = max(res_val, key=res_val.get)
print("Recall@10 en val por ventana:", {str(k): round(v, 4) for k, v in res_val.items()}, "→ mejor", w_best)

tr_full = pd.concat([tr, va]); us = te.user_id.unique()
usuarios = load_users(cfg.size)
if usuarios is not None:
    segmento = pd.cut(usuarios.set_index("user_id").age, [0, 18, 25, 35, 45, 120], labels=False).to_dict()
else:
    segmento = pd.qcut(tr_full.user_id.value_counts(), 4, labels=False).to_dict()
recs = {"aleatorio": rec_aleatorio(tr_full, us), "popularidad": rec_popular(tr_full, us),
        f"popularidad reciente ({'todo el histórico' if w_best is None else f'{w_best} d'})":
            rec_popular(tr_full, us, window_days=w_best),
        "popularidad por segmento": rec_segmento(tr_full, us, segmento)}
tabla = pd.DataFrame({n: {"Recall@10": recall_at_k(r, te), "NDCG@10": ndcg_at_k(r, te)} for n, r in recs.items()}).T
assert tabla.loc["popularidad", "Recall@10"] >= 5 * tabla.loc["aleatorio", "Recall@10"], "popularidad < 5× aleatorio"
tabla.round(4)

In [ ]:
inter = dedup(ratings); inter = inter[inter.rating >= cfg.threshold]
msk = np.random.default_rng(SEED).random(len(inter)) < 0.2
tr_a, te_a = inter[~msk], drop_cold(inter[~msk], inter[msk])
contaminado = pd.concat([tr, te])
rank_c = contaminado.item_id.value_counts().index.to_numpy()
vistos = tr.groupby("user_id").item_id.agg(set)
recs_c = {u: _topk(rank_c, vistos.get(u, set()), 10) for u in te.user_id.unique()}
leak = {"temporal (correcto)": recall_at_k(rec_popular(tr, te.user_id.unique()), te),
        "aleatorio 80/20": recall_at_k(rec_popular(tr_a, te_a.user_id.unique()), te_a),
        "temporal + popularidad\ncon test (leakage)": recall_at_k(recs_c, te)}
pd.Series(leak).plot.bar(rot=0, color=["#4C72B0", "#9E9E9E", "#C44E52"], figsize=(7, 3.2),
                         title="Popularidad: Recall@10 según el protocolo")
plt.ylabel("Recall@10"); plt.tight_layout(); plt.show()
leak

**Lectura**: usar el test para calcular la popularidad le da al modelo información de lo que se va a ver (el efecto aquí es pequeño porque la "feature" es una sola; con muchas *features* agregadas es enorme). El split aleatorio mide una tarea distinta (rellenar huecos del pasado) con usuarios y distribución distintos, así que su número no es comparable con el temporal — y no dice nada de cómo funcionará el modelo mañana. Moraleja: **el protocolo forma parte del resultado**.

### Empaquetar el pipeline como módulo
La versión canónica del curso es `cinematch_data.py`. La escribimos en disco (celdas `%%writefile`) para que el resto del curso la importe, y comprobamos que **tu pipeline y el canónico producen exactamente los mismos splits**.

In [ ]:
%%writefile cinematch_data.py
"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).

Interfaz estable que reutilizan los módulos posteriores:

    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"
    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)
    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)
    inter = filter_k_core(inter, min_user=5, min_item=5)
    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)
    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test
    train, test = random_split(inter, test_frac=0.2)
    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse
    recs = recommend_popular(train, users=test.user_id.unique(), k=10)
    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado

Convenciones de columnas: user_id (int), item_id (int), rating (float),
timestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").

Si la descarga de GroupLens falla (sin red), `load_movielens` genera un dataset
sintético con el mismo esquema (`make_synthetic_movielens`) y avisa.
"""
from __future__ import annotations

import io
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

BASE_URL = "https://files.grouplens.org/datasets/movielens"
SIZES = {
    "100k": "ml-100k",
    "1m": "ml-1m",
    "latest-small": "ml-latest-small",
    "25m": "ml-25m",
    "32m": "ml-32m",
}
GENRES_100K = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
    "Romance", "Sci-Fi", "Thriller", "War", "Western",
]


# ----------------------------------------------------------------------------
# Descarga y carga
# ----------------------------------------------------------------------------
def download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:
    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""
    if size not in SIZES:
        raise ValueError(f"size debe ser uno de {list(SIZES)}")
    name = SIZES[size]
    data_dir = Path(data_dir)
    folder = data_dir / name
    if folder.exists():
        return folder
    data_dir.mkdir(parents=True, exist_ok=True)
    url = f"{BASE_URL}/{name}.zip"
    print(f"Descargando {url} ...")
    with urllib.request.urlopen(url, timeout=60) as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)
    return folder

In [ ]:
%%writefile -a cinematch_data.py


def _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:
    year = titles.str.extract(r"\((\d{4})\)\s*$")[0].astype("float")
    clean = titles.str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    return clean, year


def _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if size == "100k":
        ratings = pd.read_csv(folder / "u.data", sep="\t",
                              names=["user_id", "item_id", "rating", "timestamp"])
        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K
        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")
        flags = raw[GENRES_100K].values.astype(bool)
        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]
        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),
                              "genres": genres})
    elif size == "1m":
        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",
                              names=["user_id", "item_id", "rating", "timestamp"])
        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",
                            names=["item_id", "title", "genres"], encoding="latin-1")
    else:
        ratings = pd.read_csv(folder / "ratings.csv").rename(
            columns={"userId": "user_id", "movieId": "item_id"})
        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})
    items["title"], items["year"] = _split_title_year(items["title"].astype(str))
    return ratings, items[["item_id", "title", "year", "genres"]]


def load_movielens(size: str = "100k", data_dir: str | Path = "data",
                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Devuelve (ratings, items) con el esquema estándar de CineMatch.

    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.
    items:   item_id, title, year, genres ("Action|Comedy").
    """
    try:
        folder = download_movielens(size, data_dir)
        ratings, items = _read_folder(size, folder)
    except Exception as e:  # sin red, URL caída, etc.
        if not synthetic_fallback:
            raise
        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "
                      "Usando datos SINTÉTICOS con el mismo esquema.")
        ratings, items = make_synthetic_movielens()
    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",
                              "rating": "float32", "timestamp": "int64"})
    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")
    return ratings.reset_index(drop=True), items.reset_index(drop=True)


def load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:
    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""
    try:
        folder = download_movielens(size, data_dir)
    except Exception:
        return None
    if size == "100k":
        return pd.read_csv(folder / "u.user", sep="|",
                           names=["user_id", "age", "gender", "occupation", "zip"])
    if size == "1m":
        return pd.read_csv(folder / "users.dat", sep="::", engine="python",
                           names=["user_id", "gender", "age", "occupation", "zip"])
    return None

In [ ]:
%%writefile -a cinematch_data.py


def make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,
                             n_ratings: int = 100_000, seed: int = 42
                             ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,
    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""
    rng = np.random.default_rng(seed)
    genres = GENRES_100K[1:]
    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K
    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada
    item_genre = rng.integers(0, len(genres), n_items)
    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1
    rng.shuffle(item_pop)
    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,
                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos
    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta
    activity = rng.lognormal(0, 1.0, n_users)
    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)
    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)
    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)
    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras
    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes
    rows = []
    for u in range(n_users):
        start = user_start[u]
        end = min(t1, start + rng.uniform(1, 120) * 86_400)
        avail = item_arrival <= end
        n = int(min(activity[u], avail.sum() // 2))
        ts = np.sort(rng.uniform(start, end, n))
        age_days = np.maximum(0, end - item_arrival) / 86_400
        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento
        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail
        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())
        ts = np.maximum(ts, item_arrival[chosen] + 3600)
        aff = taste[u][item_genre[chosen]] * len(genres)
        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]
                             + rng.normal(0, 0.7, n)), 1, 5)
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,
                                  "rating": r, "timestamp": ts.astype(np.int64)}))
    ratings = pd.concat(rows, ignore_index=True)
    second = rng.integers(0, len(genres), n_items)
    items = pd.DataFrame({
        "item_id": np.arange(1, n_items + 1),
        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],
        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),
        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"
                   for a, b in zip(item_genre, second)],
    })
    return ratings, items


# ----------------------------------------------------------------------------
# Limpieza y señales
# ----------------------------------------------------------------------------
def to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:
    """Convierte ratings explícitos en interacciones implícitas positivas
    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""
    out = ratings[ratings["rating"] >= threshold].copy()
    return out.reset_index(drop=True)


def dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:
    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""
    return (df.sort_values("timestamp", kind="stable")
              .drop_duplicates(["user_id", "item_id"], keep=keep)
              .reset_index(drop=True))

In [ ]:
%%writefile -a cinematch_data.py


def filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,
                  max_iter: int = 100) -> pd.DataFrame:
    """k-core iterativo: repite hasta que todo usuario tenga >= min_user
    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""
    for _ in range(max_iter):
        n0 = len(df)
        ic = df["item_id"].map(df["item_id"].value_counts())
        df = df[ic >= min_item]
        uc = df["user_id"].map(df["user_id"].value_counts())
        df = df[uc >= min_user]
        if len(df) == n0:
            break
    return df.reset_index(drop=True)


# ----------------------------------------------------------------------------
# Splits
# ----------------------------------------------------------------------------
def _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:
    if users:
        other = other[other["user_id"].isin(train["user_id"].unique())]
    if items:
        other = other[other["item_id"].isin(train["item_id"].unique())]
    return other.reset_index(drop=True)


def random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,
                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""
    mask = np.random.default_rng(seed).random(len(df)) < test_frac
    train, test = df[~mask].reset_index(drop=True), df[mask]
    return train, _drop_cold(train, test, drop_cold, drop_cold)


def leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1
                        ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.
    Ojo: no hay un corte temporal global → el train de un usuario puede contener
    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""
    df = df.sort_values(["user_id", "timestamp"], kind="stable")
    rank_from_end = df.groupby("user_id").cumcount(ascending=False)
    n_user = df.groupby("user_id")["item_id"].transform("size")
    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)
    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)


def temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,
                   drop_cold_users: bool = True, drop_cold_items: bool = True
                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo
    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a
    producción: entrenas con todo lo anterior a T y sirves después de T.
    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""
    ts = df["timestamp"]
    t_test = ts.quantile(1 - test_frac)
    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test
    train = df[ts < t_val].reset_index(drop=True)
    val = df[(ts >= t_val) & (ts < t_test)]
    test = df[ts >= t_test]
    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)
    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)
    seen = pd.concat([train, val])
    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)
    return train, val, test

In [ ]:
%%writefile -a cinematch_data.py


# ----------------------------------------------------------------------------
# Matrices dispersas
# ----------------------------------------------------------------------------
def build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:
    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""
    users = np.sort(df["user_id"].unique())
    items = np.sort(df["item_id"].unique())
    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})


def to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,
           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:
    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).
    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""
    if user2idx is None or item2idx is None:
        user2idx, item2idx = build_mappings(df)
    r = df["user_id"].map(user2idx)
    c = df["item_id"].map(item2idx)
    ok = r.notna() & c.notna()
    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \
        else df.loc[ok, value_col].to_numpy(np.float32)
    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),
                      shape=(len(user2idx), len(item2idx)))
    X.sum_duplicates()
    return X, user2idx, item2idx


# ----------------------------------------------------------------------------
# Baselines
# ----------------------------------------------------------------------------
def _seen_by_user(train: pd.DataFrame) -> dict:
    return train.groupby("user_id")["item_id"].agg(set).to_dict()


def _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:
    out = []
    for it in ranked:
        if it not in seen:
            out.append(int(it))
            if len(out) == k:
                break
    return out


def popularity_ranking(train: pd.DataFrame, window_days: float | None = None,
                       half_life_days: float | None = None) -> np.ndarray:
    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.
    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""
    df = train
    t_max = df["timestamp"].max()
    if window_days is not None:
        df = df[df["timestamp"] >= t_max - window_days * 86_400]
    if half_life_days is not None:
        age = (t_max - df["timestamp"]) / 86_400
        w = 0.5 ** (age / half_life_days)
        scores = w.groupby(df["item_id"]).sum()
    else:
        scores = df["item_id"].value_counts()
    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()


def recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,
                      window_days: float | None = None,
                      half_life_days: float | None = None) -> dict:
    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""
    ranked = popularity_ranking(train, window_days, half_life_days)
    seen = _seen_by_user(train) if exclude_seen else {}
    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}

In [ ]:
%%writefile -a cinematch_data.py


def recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,
                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:
    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).
    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""
    seg = train["user_id"].map(segment_of)
    global_rank = popularity_ranking(train)
    ranks = {}
    for s, g in train.groupby(seg):
        if len(g) >= min_segment_events:
            r = g["item_id"].value_counts().index.to_numpy()
            # completar con el ranking global para no quedarnos cortos
            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])
    seen = _seen_by_user(train) if exclude_seen else {}
    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)
            for u in users}


def recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,
                     exclude_seen: bool = True) -> dict:
    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""
    rng = np.random.default_rng(seed)
    items = train["item_id"].unique()
    seen = _seen_by_user(train) if exclude_seen else {}
    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}


# ----------------------------------------------------------------------------
# Persistencia
# ----------------------------------------------------------------------------
def save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:
    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, f in frames.items():
        f.to_parquet(out_dir / f"{name}.parquet", index=False)
    return out_dir


def load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:
    out_dir = Path(out_dir)
    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names
            if (out_dir / f"{n}.parquet").exists()}


# ----------------------------------------------------------------------------
# Pipeline completo de CineMatch (una llamada)
# ----------------------------------------------------------------------------
def prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,
                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,
                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:
    """Pipeline estándar del curso:
    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal
    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test
    restringidos a usuarios e ítems conocidos en train.

    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si
    out_dir no es None, los guarda en parquet."""
    ratings, items = load_movielens(size, data_dir)
    inter = dedup_interactions(ratings)
    if threshold is not None:
        inter = to_implicit(inter, threshold)
    train, val, test = temporal_split(inter, val_frac, test_frac,
                                      drop_cold_users=False, drop_cold_items=False)
    train = filter_k_core(train, min_user, min_item)
    val = _drop_cold(train, val, True, True)
    test = _drop_cold(train, test, True, True)
    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}
    if out_dir is not None:
        save_splits(out_dir, train=train, val=val, test=test, items=items)
    return out

In [ ]:
import importlib

import cinematch_data
importlib.reload(cinematch_data)
canon = cinematch_data.prepare_cinematch(cfg.size, cfg.threshold, cfg.min_user, cfg.min_item,
                                         cfg.val_frac, cfg.test_frac)
for k in ["train", "val", "test"]:
    a = splits[k].sort_values(["user_id", "item_id"]).reset_index(drop=True)
    b = canon[k][a.columns].sort_values(["user_id", "item_id"]).reset_index(drop=True)
    pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("✔ tu pipeline == cinematch_data.prepare_cinematch")

---
## 🚀 Retos extra
1. **Escala**: ejecuta con `SCALE="full"` (MovieLens-1M) y luego adapta el cargador a **MovieLens-25M/32M** usando `pyarrow` o **Polars**. Mide tiempo y memoria.
2. **Validación declarativa**: reescribe `validar` con **Pandera** (esquemas + checks) y hazla fallar a propósito con datos corruptos.
3. **Señal implícita realista**: genera un log de plays con `% visto` (como en la lección) y construye un positivo "visto ≥ 70 %". ¿Cuánto cambian los baselines?
4. **Split por ventanas deslizantes** (*rolling origin*): 4 cortes temporales consecutivos; reporta media y desviación de Recall@10. ¿Es estable el ranking de baselines?
5. **Versionado**: registra el dataset como artefacto en **MLflow** o **DVC** usando el `hash` de la config como versión.

## 🤔 Reflexión (producción / MLOps)
- ¿Qué parte de este pipeline correría en *batch* cada noche y cuál debería ser *streaming*?
- Si mañana el equipo de cliente cambia el evento `play` para que se dispare a los 2 segundos en vez de a los 30, ¿qué validación lo detectaría? ¿Cómo afectaría al modelo?
- ¿Cómo garantizarías *point-in-time correctness* si añadieras *features* de usuario (p. ej. "nº de películas vistas en los últimos 7 días")?